# 2. Ringing a black hole and extracting gravitational waves

A Schwarzschild black hole ($M=1$) in Kerr-Schild coordinates, with an $\ell=2, m=0$ perturbation added to the metric.
We compute $\Psi_4$, decompose it into spin-weighted harmonics on a geodesic sphere, and read it with kuibit.

The physics target is the quasi-normal mode $M\omega_{220} = 0.3737 - 0.0890\,i$. As you will see, the plain ADM system
becomes unstable before a clean ringdown can be measured. Finding out *why* is the point of this notebook, and it
motivates BSSN.

## Settings: where runs are written and read

This notebook writes runs to, and reads them from, **`OUTPUT_ROOT`**. By default
that is `<repository>/simulations`, the same place `pynr run par/<name>.par`
writes to from the shell. So you can:

* run here (`RUN_SIMULATIONS = True`), or
* run in a terminal, e.g. `pynr run par/schwarzschild_perturbed.par`, then set
  `RUN_SIMULATIONS = False` and only plot.

Change `OUTPUT_ROOT` to keep this notebook's runs elsewhere. For the shell to
use the same folder, run `PYNR_OUTPUT_DIR=<that folder> pynr run ...` (or
`pynr run ... --output-root <that folder>`).

In [ ]:
import os
from pynr import paths

OUTPUT_ROOT = paths.output_root()                   # default: <repo>/simulations (same as `pynr run`)
# OUTPUT_ROOT = os.path.expanduser("~/pynr_runs")   # ...or any folder you like
RUN = "schwarzschild_perturbed"                     # run name = output folder (default: parfile name)
FINAL_TIME = None                                   # e.g. 24.0 for a shorter run (~8 min on 12 cores)

os.environ["PYNR_OUTPUT_DIR"] = OUTPUT_ROOT
RUN_SIMULATIONS = RUN not in paths.list_runs()      # run only if it does not exist yet; set True/False to force
print("output root  :", OUTPUT_ROOT)
print("existing runs:", paths.list_runs())
print("RUN_SIMULATIONS =", RUN_SIMULATIONS)

## Run (optional; the full parameter file takes ~12-15 min on 12 cores)

In [ ]:
from pynr import Simulation

if RUN_SIMULATIONS:
    over = {"IO::out_dir": RUN, "IOBasic::outInfo_every": 40}
    if FINAL_TIME is not None:
        over["Cactus::cctk_final_time"] = FINAL_TIME
    Simulation.from_parfile("../par/schwarzschild_perturbed.par", overrides=over).run()

## Plot: $\Psi_4$ multipoles (read from disk with kuibit)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from kuibit.simdir import SimDir

sd = SimDir(paths.run_dir(RUN))
print("extraction radii:", sd.gws.radii)
r = min(sd.gws.radii)
for (l, m) in [(2, 0), (4, 0)]:
    p = sd.gws[r][(l, m)]
    plt.semilogy(p.t, np.abs(p.y.real), label=f"|Re Psi4_{l}{m}|")
tt = np.linspace(10, max(p.t), 50)
plt.semilogy(tt, 5e-4 * np.exp(-0.089 * (tt - 10)), "k--", label="QNM decay")
plt.xlabel("t / M"); plt.ylabel(f"|Psi4| at r = {r} M"); plt.legend()

## Checks and a (failing) ringdown fit

1. Symmetry: $\operatorname{Im}\Psi_4^{20}$ and the odd-$\ell$ modes should be at round-off level. Why?
2. Try to fit $A e^{-t/\tau}\cos(\omega t + \phi)$ after the burst and compare with $\omega = 0.3737$, $\tau = 11.2$.
   Then plot `sd.ts.norm2["H"]` on a log scale. When does the constraint violation start to grow, and how does
   that time compare with the QNM period?

In [ ]:
from scipy.optimize import curve_fit
p = sd.gws[r][(2, 0)]
t0, t1 = 12.0, 24.0      # adjust after looking at the plot above
msk = (p.t > t0) & (p.t < t1)
f = lambda t, A, tau, w, ph: A * np.exp(-(t - t0) / tau) * np.cos(w * (t - t0) + ph)
pp, _ = curve_fit(f, p.t[msk], p.y.real[msk], p0=[np.abs(p.y[msk]).max(), 11, 0.37, 0], maxfev=20000)
print(f"omega = {pp[2]:.4f}  (QNM 0.3737),  tau = {pp[1]:.2f}  (QNM 11.24)")
H = sd.ts.norm2["H"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(p.t, p.y.real); ax[0].plot(p.t[msk], f(p.t[msk], *pp), "--"); ax[0].set_title("fit")
ax[1].semilogy(H.t, H.y); ax[1].set_title("||H||_2")

### Things to try
* Compare the $(2,0)$ and $(4,0)$ modes as functions of time. Which one grows, and why is that a sign of a numerical
  rather than a physical mode?
* Change `Dissipation::epsdis`, `ADMEvolve::excision_radius` and the resolution. How do they affect the time at which $H$ starts growing?
* The perturbation does not satisfy the Hamiltonian constraint. How large is $H$ at $t=0$ compared with the unperturbed run?